# Final result

Global matching, the test evaluation and the group sizes of each round.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
PAIRS = ["wikidata-ror", "deqar-ror"]

In [ ]:
final = pd.read_csv("results/final.csv")
final[["round", "pair", "matcher", "split", "P", "R", "F1", "corr"]]

## Global matching (validation labels)

In [ ]:
val = final[final["split"] == "validation"].copy()
val["global"] = val["matcher"].str.split(" \\+ ").str[-1]
for rnd in sorted(val["round"].unique()):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
    for ax, pair in zip(axes, PAIRS):
        part = val[(val["round"] == rnd) & (val["pair"] == pair)].set_index("global")[["P", "R", "F1"]]
        part.plot.barh(ax=ax, width=0.8)
        ax.set_xlim(0, 1)
        ax.set_title(f"round {rnd}: {pair}")
        ax.legend(loc="lower right")
    plt.tight_layout()

## Test result of each round

The test labels are only used once per round, for the final choice.

In [ ]:
test = final[final["split"] == "test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, pair in zip(axes, PAIRS):
    test[test["pair"] == pair].set_index("round")[["P", "R", "F1"]].plot.bar(ax=ax, rot=0)
    ax.set_ylim(0, 1)
    ax.set_title(pair)
plt.tight_layout()
test[["round", "pair", "matcher", "P", "R", "F1", "corr"]]

## Group size distribution

With duplicate-free sources a group should have at most 3 records (one per source).

In [ ]:
from pathlib import Path

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, rnd in zip(axes, [1, 2, 3]):
    path = Path(f"results/group_sizes_r{rnd}.csv")
    if not path.exists():
        continue
    sizes = pd.read_csv(path)
    ax.bar(sizes["cluster_size"].astype(str).head(15), sizes["frequency"].head(15))
    ax.set_yscale("log")
    ax.set_title(f"round {rnd}: largest group {sizes['cluster_size'].max():,}")
    ax.set_xlabel("group size")
plt.tight_layout()